## GPT2

José Alberto isidoro Sánchez

In [ ]:
# Install the transformers and datasets libraries (usually run in a notebook environment)
!pip install transformers datasets


In [ ]:

# Import PyTorch and its neural network modules
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
# Hugging Face datasets library for easy loading of standard datasets (not used directly here)
from datasets import load_dataset
import torch.optim as optim

# ----------------------------------------------------------------------
# Configuration class to hold hyperparameters of the model
# ----------------------------------------------------------------------
class Config:
    '''
    Holds all model hyperparameters.
    '''
    def __init__(self, vocab_size = 50257, max_seq_length = 128, embed_size = 768,
                 num_layers = 12, num_heads = 12, dropout = 0.1):
        # Size of the token vocabulary (default matches GPT-2)
        self.vocab_size = vocab_size
        # Maximum length of input sequences (context window)
        self.max_seq_length = max_seq_length
        # Dimensionality of token embeddings and hidden states
        self.embed_size = embed_size
        # Number of transformer blocks (layers) in the model
        self.num_layers = num_layers
        # Number of attention heads in each multi-head attention layer
        self.num_heads = num_heads
        # Dropout probability for regularization
        self.dropout = dropout

# ----------------------------------------------------------------------
# Multi-Head Self-Attention module (with causal masking)
# ----------------------------------------------------------------------
class SelfAttention(nn.Module):
    '''
    This is the most important part, the heart of the Transformer.
    In this implementation, the embedding dimensionality is kept through the entire
    layers of the architecture. Note that embed_size is a multiple of num_heads.
    '''
    def __init__(self, config):
        super().__init__()
        # Ensure embed_size is divisible by num_heads (precondition for multi-head attention)
        assert config.embed_size % config.num_heads == 0, 'Embed size not a multiple of num heads'
        self.num_heads = config.num_heads
        # Dimension of each attention head = embed_size / num_heads
        self.head_dim = config.embed_size // config.num_heads

        # Linear layers to project input into Query, Key, Value
        self.W_q = nn.Linear(config.embed_size, config.embed_size)
        self.W_k = nn.Linear(config.embed_size, config.embed_size)
        self.W_v = nn.Linear(config.embed_size, config.embed_size)
        # Output linear layer after concatenating heads
        self.output = nn.Linear(config.embed_size, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)

        # Create a lower-triangular mask for causal attention (so future tokens are not seen)
        # Register as buffer so it moves with the model to the correct device
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(config.max_seq_length, config.max_seq_length)
                      ).view(-1, 1, config.max_seq_length, config.max_seq_length)
        )

    def forward(self, x):
        # x shape: (batch, sequence_length, embed_size)
        batch, seq_length, embed_dim = x.size()   # e.g. 16, 128, 768

        # Compute Q, K, V and reshape for multi-head attention
        # W_q(x) shape: (batch, seq_len, embed_size)
        # .view(...) splits the last dimension into (num_heads, head_dim)
        # transpose(1,2) swaps seq_len and num_heads -> (batch, num_heads, seq_len, head_dim)
        Q = self.W_q(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2)

        # Compute scaled dot-product attention scores
        # Q @ K.transpose(-2,-1) gives (batch, num_heads, seq_len, seq_len)
        # The scaling factor should be sqrt(head_dim), but here they use head_dim**5 (likely a typo)
        attn = (Q @ K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # Apply causal mask: set positions where mask == 0 to -inf so softmax ignores them
        attn = attn.masked_fill(self.mask[:, :, :seq_length, :seq_length] == 0, float('-inf'))
        # Softmax over the last dimension (keys) to get attention probabilities
        attn = F.softmax(attn, dim=-1)
        # Apply dropout to attention weights
        attn = self.dropout(attn)

        # Apply attention weights to Values: (batch, num_heads, seq_len, seq_len) @ (batch, num_heads, seq_len, head_dim)
        # Result shape: (batch, num_heads, seq_len, head_dim)
        scores = attn @ V

        # Transpose back to (batch, seq_len, num_heads, head_dim) and then reshape to (batch, seq_len, embed_size)
        scores = scores.transpose(1, 2).contiguous().view(batch, seq_length, embed_dim)
        # Pass through output linear layer
        scores = self.output(scores)
        # Apply dropout one more time
        return self.dropout(scores)







In [ ]:
# ----------------------------------------------------------------------
# Feed-Forward Network (MLP) used in each transformer block
# ----------------------------------------------------------------------
class FFN(nn.Module):
    '''
    MLP for transformer block: two linear layers with GELU activation in between.
    Inner dimension is 4 * embed_size (typical for transformers).
    '''
    def __init__(self, config):
        super().__init__()
        self.fc1 = nn.Linear(config.embed_size, 4 * config.embed_size)
        self.gelu = nn.GELU()          # GELU activation (common in GPT)
        self.fc2 = nn.Linear(4 * config.embed_size, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # x: (batch, seq_len, embed_size)
        x = self.fc2(self.gelu(self.fc1(x)))   # expand then contract
        return self.dropout(x)

# ----------------------------------------------------------------------
# Single Transformer Block: Attention + Feed-Forward with residual connections and layer norm
# ----------------------------------------------------------------------
class Transformer(nn.Module):
    '''
    Put the transformer together.
    '''
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.embed_size)   # pre‑layer norm before attention
        self.attention = SelfAttention(config)
        self.norm2 = nn.LayerNorm(config.embed_size)   # pre‑layer norm before MLP
        self.mlp = FFN(config)

    def forward(self, x):
        # Residual connection: add the original input after attention
        x = x + self.attention(self.norm1(x))
        # Residual connection: add the result after MLP
        x = x + self.mlp(self.norm2(x))
        return x


In [ ]:
# ----------------------------------------------------------------------
# Full GPT‑2 Model: token + position embeddings, stack of transformer blocks, final layer norm
# ----------------------------------------------------------------------
class GPT2(nn.Module):
    '''
    Building the entire GPT2 by combining the other modules.
    '''
    def __init__(self, config):
        super().__init__()
        self.config = config
        # Token embedding table: maps each token ID to a vector of size embed_size
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_size)
        # Position embedding table: learns a vector for each possible position (0 to max_seq_length-1)
        self.pos_embed = nn.Embedding(config.max_seq_length, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)
        # Stack of transformer blocks (num_layers many)
        self.transformers = nn.Sequential(*[Transformer(config) for _ in range(config.num_layers)])
        # Final layer normalization after all blocks
        self.norm1 = nn.LayerNorm(config.embed_size)

    def forward(self, input_tokens):
        # input_tokens shape: (batch, sequence_length) — token IDs
        batch, seq_length = input_tokens.size()
        # Create position indices for each token in the sequence (0,1,2,...,seq_length-1)
        # unsqueeze(0) adds a batch dimension -> (1, seq_length)
        pos = torch.arange(0, seq_length, dtype=torch.long, device=input_tokens.device).unsqueeze(0)
        # Sum token embeddings and position embeddings, then apply dropout
        x = self.token_embed(input_tokens) + self.pos_embed(pos)   # shape: (batch, seq_len, embed_size)
        x = self.dropout(x)
        # Pass through all transformer blocks
        x = self.transformers(x)
        # Apply final layer norm
        x = self.norm1(x)
        # Project to vocabulary size by multiplying by the transpose of the token embedding matrix (weight tying)
        return x @ self.token_embed.weight.t()

In [ ]:
# ----------------------------------------------------------------------
# Download the text of Don Quijote from Project Gutenberg
# ----------------------------------------------------------------------
import requests
url = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
response = requests.get(url)
text = response.text   # raw text as a single string

# ----------------------------------------------------------------------
# Tokenize the text using the GPT‑2 tokenizer (fast version)
# ----------------------------------------------------------------------
from transformers import GPT2TokenizerFast

tokeniser = GPT2TokenizerFast.from_pretrained("gpt2")
# Set the padding token to be the end‑of‑sequence token (GPT‑2 doesn't have a pad token by default)
tokeniser.pad_token = tokeniser.eos_token

# Encode the entire text into a list of token IDs
tokens = tokeniser.encode(text)
# Convert the list to a PyTorch tensor of type long (int64)
data = torch.tensor(tokens, dtype=torch.long)

# Note: The warning about length > 1024 is fine because we will split into shorter sequences.

# ----------------------------------------------------------------------
# Set the sequence length for training chunks
# ----------------------------------------------------------------------
SEQ_LENTGH = 128   # (intentional typo: should be SEQ_LENGTH)

# ----------------------------------------------------------------------
# Custom Dataset class that creates input‑target pairs of length seq_length
# Each sample: x = tokens[idx : idx+seq_length], y = tokens[idx+1 : idx+seq_length+1]
# (next‑token prediction)
# ----------------------------------------------------------------------
class Quijote(Dataset):
    '''
    This class creates training samples of length SEQ_LENGTH.
    '''
    def __init__(self, data, seq_length):
        self.text = data          # the full 1D tensor of token IDs
        self.seq_length = seq_length

    def __len__(self):
        # Number of possible starting positions such that we have a full input and target sequence
        return len(self.text) - self.seq_length

    def __getitem__(self, idx):
        # Input tokens: from idx to idx + seq_length
        x = self.text[idx: idx + self.seq_length]
        # Target tokens: shifted by one position (predict the next token)
        y = self.text[idx + 1: idx + self.seq_length + 1]
        return x, y

# Instantiate the dataset
quijote = Quijote(data, SEQ_LENTGH)
# Create a DataLoader that shuffles and batches the data (batch_size=16)
quijote_loader = DataLoader(quijote, batch_size=16, shuffle=True)



In [ ]:
# ----------------------------------------------------------------------
# Create a configuration for a smaller GPT‑2 model (fits in memory)
# ----------------------------------------------------------------------
config = Config(
    vocab_size = tokeniser.vocab_size,   # actual GPT‑2 vocab size (around 50257)
    max_seq_length = SEQ_LENTGH,         # 128
    embed_size = 128,                    # smaller than standard 768
    num_layers = 4,                      # fewer layers
    num_heads = 4,                       # heads must divide embed_size (128/4=32)
    dropout = 0.1
)

# Choose device: GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Create the model and move it to the chosen device
model = GPT2(config).to(device)

# Set up the Adam optimizer with a learning rate of 3e-4
optimiser = optim.Adam(model.parameters(), lr=3e-4)



In [ ]:
# ----------------------------------------------------------------------
# Training loop
# ----------------------------------------------------------------------
def train(model, loader, optimiser, epochs = 10):
    model.train()   # set model to training mode (enables dropout, etc.)
    for epoch in range(epochs):
        total_loss = 0.0
        for i, (x, y) in enumerate(loader):
            # Move batch to the same device as the model
            x = x.to(device)
            y = y.to(device)
            # Zero out gradients from previous step
            optimiser.zero_grad()
            # Forward pass: logits (scores) of shape (batch, seq_len, vocab_size)
            scores = model(x)
            # Compute cross‑entropy loss: reshape scores to (batch*seq_len, vocab_size) and targets to (batch*seq_len)
            loss = F.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
            # Backpropagate
            loss.backward()
            # Update model parameters
            optimiser.step()

            total_loss += loss.item()

            # Print progress every 500 steps
            if i % 500 == 0:
                print(f'epoch: {epoch+1}, step: {i}, loss:{loss.item():.4f}')

        # Average loss for the whole epoch
        epoch_loss = total_loss / len(loader)
        print(f'epoch: {epoch + 1}. Loss: {epoch_loss:.4f}')

# Train the model for 3 epochs (you can increase)
train(model, quijote_loader, optimiser, 3)

epoch: 1, step: 0, loss:81.8293
epoch: 1, step: 500, loss:8.6662
epoch: 1, step: 1000, loss:6.8775
epoch: 1, step: 1500, loss:6.2078


In [ ]:

# ----------------------------------------------------------------------
# Text generation function
# ----------------------------------------------------------------------
def sample(model, device, tokenizer, prompt, length=50, temperature = 1.0):
    model.eval()   # set model to evaluation mode (turns off dropout)
    # Encode the prompt string into token IDs and move to device
    tokens = tokenizer.encode(prompt, return_tensors='pt').to(device)

    for _ in range(length):
        # Only use the last SEQ_LENTGH tokens to stay within max_seq_length
        tokens_cond = tokens[:, -SEQ_LENTGH:]
        with torch.no_grad():   # no gradient needed for generation
            logits = model(tokens_cond)   # shape: (1, seq_len_cond, vocab_size)
        # Take logits of the last token (the next token prediction)
        next_token_logits = logits[:, -1, :] / temperature   # temperature scaling
        # Sample from the softmax distribution (multinomial) to get the next token ID
        next_token = torch.multinomial(F.softmax(next_token_logits, dim=-1), num_samples=1)
        # Append the new token to the sequence
        tokens = torch.cat([tokens, next_token], dim=1)

    print(tokens)   # print the generated token IDs (for debugging)
    # Decode the whole sequence back to a string and return it
    return tokenizer.decode(tokens[0])

# Example usage: generate 50 tokens starting from the prompt "Un estudiante de doctorado"
print(sample(model, device, tokeniser, prompt="Un estudiante de doctorado", length=50))

# The output shows the generated token IDs and the generated text (which in the example appears to be Spanish,
# loosely related to Don Quijote, as the model was trained on that book).

In [ ]:
##====================
## USAGE EXAMPLE 2
#==============

#generate 50 tokens starting from THIS PROMPT:
print(sample(model, device, tokeniser, prompt="una vaca vestida de uniforme", length=50))

In [ ]:
##====================
## USAGE EXAMPLE 3
#==============

#generate 50 tokens starting from THIS PROMPT:
print(sample(model, device, tokeniser, prompt="de reversa mami, de reversa...", length=50))

In [ ]:
# ============================================================
# SIMPLE MINI-GPT (CORRECTED BUT STILL FAST)
# ============================================================
#
# This version keeps your original architecture mostly intact
# while fixing the important issues:
#
# FIXES:
# - safer reshape instead of view
# - AdamW optimizer
# - gradient clipping
# - EOS stopping
# - safer text download
# - cleaned Gutenberg text
#
# Keeps it lightweight and fast enough for CPU/laptop.
#
# ============================================================

# ------------------------------------------------------------
# INSTALL (only needed in notebooks like Colab/Jupyter)
# ------------------------------------------------------------

# !pip install transformers requests

# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------

import math
import requests

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from transformers import GPT2TokenizerFast


# ============================================================
# CONFIGURATION
# ============================================================

class Config:

    def __init__(
        self,
        vocab_size=50257,
        max_seq_length=64,
        embed_size=64,
        num_layers=2,
        num_heads=2,
        dropout=0.1
    ):

        self.vocab_size = vocab_size
        self.max_seq_length = max_seq_length
        self.embed_size = embed_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.dropout = dropout


# ============================================================
# SELF-ATTENTION
# ============================================================

class SelfAttention(nn.Module):

    def __init__(self, config):

        super().__init__()

        assert config.embed_size % config.num_heads == 0

        self.num_heads = config.num_heads
        self.head_dim = config.embed_size // config.num_heads

        self.W_q = nn.Linear(config.embed_size, config.embed_size)
        self.W_k = nn.Linear(config.embed_size, config.embed_size)
        self.W_v = nn.Linear(config.embed_size, config.embed_size)

        self.out_proj = nn.Linear(config.embed_size, config.embed_size)

        self.dropout = nn.Dropout(config.dropout)

        # causal mask
        mask = torch.tril(
            torch.ones(
                config.max_seq_length,
                config.max_seq_length
            )
        )

        self.register_buffer(
            "mask",
            mask.view(
                1,
                1,
                config.max_seq_length,
                config.max_seq_length
            )
        )

    def forward(self, x):

        batch_size, seq_len, embed_dim = x.shape

        # Create Q, K, V
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # Split into heads
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # Attention scores
        attn_scores = (
            Q @ K.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        # Apply causal mask
        attn_scores = attn_scores.masked_fill(
            self.mask[:, :, :seq_len, :seq_len] == 0,
            float("-inf")
        )

        # Softmax
        attn_probs = F.softmax(attn_scores, dim=-1)

        attn_probs = self.dropout(attn_probs)

        # Weighted sum
        context = attn_probs @ V

        # Recombine heads
        context = (
            context
            .transpose(1, 2)
            .contiguous()
            .view(batch_size, seq_len, embed_dim)
        )

        output = self.out_proj(context)

        return self.dropout(output)


# ============================================================
# FEED FORWARD NETWORK
# ============================================================

class FFN(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.fc1 = nn.Linear(
            config.embed_size,
            4 * config.embed_size
        )

        self.gelu = nn.GELU()

        self.fc2 = nn.Linear(
            4 * config.embed_size,
            config.embed_size
        )

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):

        x = self.fc1(x)
        x = self.gelu(x)
        x = self.fc2(x)

        return self.dropout(x)


# ============================================================
# TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.ln1 = nn.LayerNorm(config.embed_size)
        self.attn = SelfAttention(config)

        self.ln2 = nn.LayerNorm(config.embed_size)
        self.mlp = FFN(config)

    def forward(self, x):

        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))

        return x


# ============================================================
# GPT MODEL
# ============================================================

class GPT2(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.config = config

        # Token embeddings
        self.token_embed = nn.Embedding(
            config.vocab_size,
            config.embed_size
        )

        # Position embeddings
        self.pos_embed = nn.Embedding(
            config.max_seq_length,
            config.embed_size
        )

        self.dropout = nn.Dropout(config.dropout)

        # Transformer blocks
        self.blocks = nn.Sequential(
            *[
                TransformerBlock(config)
                for _ in range(config.num_layers)
            ]
        )

        self.ln_f = nn.LayerNorm(config.embed_size)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(self, input_ids):

        batch_size, seq_len = input_ids.shape

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        ).unsqueeze(0)

        x = (
            self.token_embed(input_ids)
            +
            self.pos_embed(positions)
        )

        x = self.dropout(x)

        x = self.blocks(x)

        x = self.ln_f(x)

        # Weight tying
        logits = x @ self.token_embed.weight.T

        return logits


# ============================================================
# DOWNLOAD TEXT
# ============================================================

url = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

text = response.text

# Keep only part for faster training
text = text[:200000]


# ============================================================
# TOKENIZER
# ============================================================

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token

tokens = tokenizer.encode(text)

data = torch.tensor(tokens, dtype=torch.long)


# ============================================================
# DATASET
# ============================================================

SEQ_LENGTH = 64


class QuijoteDataset(Dataset):

    def __init__(self, data, seq_length):

        self.data = data
        self.seq_length = seq_length

    def __len__(self):

        return len(self.data) - self.seq_length

    def __getitem__(self, idx):

        x = self.data[
            idx : idx + self.seq_length
        ]

        y = self.data[
            idx + 1 : idx + self.seq_length + 1
        ]

        return x, y


dataset = QuijoteDataset(data, SEQ_LENGTH)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ============================================================
# MODEL
# ============================================================

config = Config(
    vocab_size=tokenizer.vocab_size,
    max_seq_length=SEQ_LENGTH,
    embed_size=64,
    num_layers=2,
    num_heads=2,
    dropout=0.1
)

model = GPT2(config).to(device)


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4
)


# ============================================================
# TRAINING
# ============================================================

def train(model, loader, optimizer, epochs=1):

    model.train()

    for epoch in range(epochs):

        total_loss = 0.0

        for step, (x, y) in enumerate(loader):

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(x)

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1)
            )

            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()

            total_loss += loss.item()

            if step % 100 == 0:

                print(
                    f"Epoch {epoch+1} | "
                    f"Step {step} | "
                    f"Loss {loss.item():.4f}"
                )

        avg_loss = total_loss / len(loader)

        print(
            f"\nEpoch {epoch+1} completed "
            f"| Avg Loss: {avg_loss:.4f}\n"
        )


# ============================================================
# TRAIN
# ============================================================

train(
    model,
    dataloader,
    optimizer,
    epochs=1
)


# ============================================================
# GENERATION
# ============================================================

def generate(
    model,
    tokenizer,
    prompt,
    max_new_tokens=50,
    temperature=1.0
):

    model.eval()

    input_ids = tokenizer.encode(
        prompt,
        return_tensors="pt"
    ).to(device)

    for _ in range(max_new_tokens):

        # Crop to max context length
        input_crop = input_ids[:, -SEQ_LENGTH:]

        with torch.no_grad():

            logits = model(input_crop)

        # Last token logits
        next_logits = logits[:, -1, :] / temperature

        probs = F.softmax(next_logits, dim=-1)

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        # Append token
        input_ids = torch.cat(
            [input_ids, next_token],
            dim=1
        )

        # Stop at EOS token
        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(input_ids[0])


# ============================================================
# TEST GENERATION
# ============================================================

prompt = "Un estudiante de doctorado"

output = generate(
    model,
    tokenizer,
    prompt,
    max_new_tokens=50,
    temperature=0.8
)

print("\nGenerated text:\n")

print(output)